# 01. Выгрузка и первичная подготовка данных

В этом ноутбуке подготовим исторические данные 

По условию задания необходимо:
- собрать исторические OHLCV данные,
- подготовить данные как минимум за 6 месяцев для baseline-модели и более чем за 2 года для RL-агента.

В качестве торговой пары используем:
- **BTC-USDT** — спотовый инструмент,
- **XBTUSDTM** — фьючерсный контракт на тот же актив.

Далее выгрузим данные через модуль `data_loader.py`, проверим их структуру и построим базовые графики, которые нужны для последующего анализа стратегии.

In [6]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
src_path = str((repo_root / "src").resolve())
config_path = repo_root / "config" / "basis_strategy_config.json"

if src_path not in sys.path:
    sys.path.insert(0, src_path)

from crypto_rl_bot.data_loader import download_history
from crypto_rl_bot.config import load_config

print("Корень проекта:", repo_root)
print("Путь к конфигу:", config_path)
print("Текущая рабочая папка:", Path.cwd())

Корень проекта: /Users/coyc_kari/Final_project_MLfr_team5
Путь к конфигу: /Users/coyc_kari/Final_project_MLfr_team5/config/basis_strategy_config.json
Текущая рабочая папка: /Users/coyc_kari/Final_project_MLfr_team5/notebooks


## Загрузим конфигурацию проекта

Сначала прочитаем параметры из конфигурационного файла, чтобы убедиться, что символы, таймфрейм и горизонты заданы корректно.

In [7]:
cfg = load_config(config_path)

print("Спот:", cfg.data.spot_symbol)
print("Фьючерс:", cfg.data.futures_symbol)
print("Интервал:", cfg.data.interval)
print("Горизонт baseline, месяцев:", cfg.data.baseline_history_months)
print("Горизонт RL, лет:", cfg.data.rl_history_years)
print("Директория raw:", cfg.data.raw_dir)
print("Директория processed:", cfg.data.processed_dir)


Спот: BTC-USDT
Фьючерс: XBTUSDTM
Интервал: 1day
Горизонт baseline, месяцев: 6
Горизонт RL, лет: 2
Директория raw: data/raw
Директория processed: data/processed


## Выгрузим данные для baseline

Сначала соберём датасет для baseline-модели. По условию проекта для базовой модели нужен горизонт 6 месяцев.

In [8]:
baseline_result = download_history(
    config_path=str(config_path),
    dataset="baseline",
)

baseline_spot_df = baseline_result["data"]["spot"].copy()
baseline_fut_df = baseline_result["data"]["futures"].copy()
baseline_df = baseline_result["data"]["merged"].copy()

print("Период baseline:", baseline_result["start_dt"], "->", baseline_result["end_dt"])
print("Число строк в споте:", len(baseline_spot_df))
print("Число строк во фьючерсе:", len(baseline_fut_df))
print("Число строк после объединения:", len(baseline_df))

display(baseline_df.head())

Период baseline: 2025-09-30 00:00:00+00:00 -> 2026-03-30 00:00:00+00:00
Число строк в споте: 181
Число строк во фьючерсе: 182
Число строк после объединения: 181


,timestamp,spot_open,spot_high,spot_low,spot_close,spot_volume,fut_open,fut_high,fut_low,fut_close,fut_volume
0,2025-09-30 00:00:00+00:00,114312.8,114781.5,112668.8,114052.4,3574.251722,114313.6,114813.8,112668.0,114044.0,4945976.0
1,2025-10-01 00:00:00+00:00,114052.4,118628.2,113967.0,118579.9,4236.283215,114044.9,118649.9,113943.0,118579.2,6188185.0
2,2025-10-02 00:00:00+00:00,118579.9,121004.7,118274.0,120528.6,4569.650038,118589.8,121021.8,118262.9,120507.1,4354720.0
3,2025-10-03 00:00:00+00:00,120528.6,123870.3,119244.5,122222.1,5716.898416,120507.2,123901.5,119229.0,122236.0,6939246.0
4,2025-10-04 00:00:00+00:00,122236.3,122793.2,121515.4,122381.3,2700.548542,122240.0,122819.0,121507.5,122375.7,2603695.0


## Выгрузим данные для RL-агента

Теперь соберём расширенный датасет для RL-агента. По условию задания для RL нужен более длинный горизонт — 2 года.

In [9]:
rl_result = download_history(
    config_path=str(config_path),
    dataset="rl",
)

rl_spot_df = rl_result["data"]["spot"].copy()
rl_fut_df = rl_result["data"]["futures"].copy()
rl_df = rl_result["data"]["merged"].copy()

print("Период RL:", rl_result["start_dt"], "->", rl_result["end_dt"])
print("Число строк в споте:", len(rl_spot_df))
print("Число строк во фьючерсе:", len(rl_fut_df))
print("Число строк после объединения:", len(rl_df))

display(rl_df.head())

Период RL: 2024-03-30 00:00:00+00:00 -> 2026-03-30 00:00:00+00:00
Число строк в споте: 730
Число строк во фьючерсе: 731
Число строк после объединения: 730


,timestamp,spot_open,spot_high,spot_low,spot_close,spot_volume,fut_open,fut_high,fut_low,fut_close,fut_volume
0,2024-03-30 00:00:00+00:00,69846.2,70331.0,69528.1,69570.9,1012.860448,69864.6,70400.0,69566.3,69610.7,2165716.0
1,2024-03-31 00:00:00+00:00,69578.4,71349.0,69553.4,71264.5,1171.486226,69615.7,71478.5,69603.5,71379.7,2858788.0
2,2024-04-01 00:00:00+00:00,71257.7,71274.8,68039.1,69653.0,3362.179645,71369.9,71369.9,68109.2,69662.3,7755603.0
3,2024-04-02 00:00:00+00:00,69653.0,69677.5,64597.5,65482.9,5617.674202,69654.7,69682.0,64579.9,65475.8,14941047.0
4,2024-04-03 00:00:00+00:00,65473.1,66890.9,64455.7,65959.2,3870.986185,65474.8,66952.6,64474.2,65955.9,7495847.0


## Проверим, куда сохранились данные

Помимо загрузки в память, модуль `data_loader.py` сохраняет данные в `data/raw`. Посмотрим на пути до файлов.

In [10]:
print("Файлы baseline:")
for key, value in baseline_result["paths"].items():
    print(f"  {key}: {value}")

print("\nФайлы RL:")
for key, value in rl_result["paths"].items():
    print(f"  {key}: {value}")

Файлы baseline:
  spot_parquet: /Users/coyc_kari/Final_project_MLfr_team5/data/raw/baseline_btcusdt_xbtusdtm_1day_20250930_20260330_spot.parquet
  futures_parquet: /Users/coyc_kari/Final_project_MLfr_team5/data/raw/baseline_btcusdt_xbtusdtm_1day_20250930_20260330_futures.parquet
  merged_parquet: /Users/coyc_kari/Final_project_MLfr_team5/data/raw/baseline_btcusdt_xbtusdtm_1day_20250930_20260330_merged.parquet
  meta_csv: /Users/coyc_kari/Final_project_MLfr_team5/data/raw/baseline_btcusdt_xbtusdtm_1day_20250930_20260330_meta.csv

Файлы RL:
  spot_parquet: /Users/coyc_kari/Final_project_MLfr_team5/data/raw/rl_btcusdt_xbtusdtm_1day_20240330_20260330_spot.parquet
  futures_parquet: /Users/coyc_kari/Final_project_MLfr_team5/data/raw/rl_btcusdt_xbtusdtm_1day_20240330_20260330_futures.parquet
  merged_parquet: /Users/coyc_kari/Final_project_MLfr_team5/data/raw/rl_btcusdt_xbtusdtm_1day_20240330_20260330_merged.parquet
  meta_csv: /Users/coyc_kari/Final_project_MLfr_team5/data/raw/rl_btcusdt_xb